In [1]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [2]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [3]:
from src.demos.ekg.retriever_tool_factory import PydanticRag

KV_STORE = "file"


vector_store_factory = PydanticRag.get_vector_store_factory()


rag = PydanticRag(
    model_definition=test_schema,
    vector_store_factory=vector_store_factory,
    llm_id=None,
    kv_store_id=KV_STORE,
)

vector_store_factory.delete_collection()

2025-08-18 16:46:50.602 | DEBUG    | src.ai_core.llm_factory:get_llm:532 - get LLM:'kimi_k2_openrouter' -extra: {'temperature': 0.0}


src/ai_core/llm_factory.py:314 LlmFactory.model_factory
    common_params: {
        'temperature': 0.0,
        'cache': None,
        'seed': 42,
        'max_retries': 2,
        'streaming': False,
    } (dict) len=5


2025-08-18 16:46:50.884 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7a79267c5b20>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-18 16:46:50.921 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:88 - pgvector vector table created: table_name='pydantic_fields_qwen3_06b' schema_name='public'
2025-08-18 16:46:50.922 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:90 - Hybrid search configured with TSV column: content_tsv


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7a79229a5a30> (PGVectorStore)


2025-08-18 16:46:50.942 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-18 16:46:50.943 | DEBUG    | src.ai_core.vector_store_factory:get:230 - get vector store  : PgVector/pydantic_fields_qwen3_06b cache_embeddings: False
2025-08-18 16:46:50.944 | INFO     | src.ai_core.vector_store_factory:delete_collection:321 - drop file public.pydantic_fields_qwen3_06b


In [5]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / doc_id
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


2025-08-18 16:53:14.486 | DEBUG    | src.utils.pydantic.kv_store:load_object_from_kvstore:89 - read 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json.json' from KV store


Structured result:
RainbowProjectAnalysis(
    identification=ProjectIdentification(
        name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE',
        opportunity_id='9000559500',
        customer='CNES',
        customer_segment='Aerospace',
        status='Solution Review',
        start_date='2019-04-01',
        end_date='2022-03-31'
    ),
    description=ProjectDescription(
        objectives=[
            'Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their 
image-processing chains',
            'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'
        ],
        scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award 
framework)',
        success_metrics=['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'],
        differentiators=[
            'Existing incumbent on PEPS',
            'Deep Earth-observation domain knowledge',
            'MUNDI platform synergies'
        ]
    ),
    team=[
        PersonRole(name='Marc Ferrer', role='Global Client Leader', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Aurore Dorez',
            role='Sales Lead / Deal Maker',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Barthélémy Marti', role='Bid Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Olivier Rondeau',
            role='Solution Manager / Contract Executive',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Caroline Jaulin', role='Finance Lead', organization='Atos', contact_type='internal'),
        PersonRole(name='Danièle Phankongsy', role='Deal Lawyer', organization='Atos', contact_type='internal'),
        PersonRole(name='Sonia Gouel', role='Project Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Pierre Bourrousse',
            role='Strategy Achat Sponsor',
            organization='CNES',
            contact_type='external'
        ),
        PersonRole(name='Gérard Lassalle-Valler', role='Sponsor', organization='CNES', contact_type='external'),
        PersonRole(
            name='Sylvia Sylvander',
            role='CP CNES Décideur',
            organization='CNES',
            contact_type='external'
        )
    ],
    delivery=DeliveryInfo(
        business_lines=['B1P S BS Toulouse'],
        locations=['Toulouse, France'],
        partners=[
            'ACS (subcontractor for Venus VIP maintenance, imposed by CNES)',
            'PIXSTART (subcontractor for THEIA MUSCATE first-year fixes and knowledge transfer)'
        ],
        technologies=[
            'VENUS VIP image-processing chains',
            'PEPS Sentinel data exploitation platform',
            'THEIA MUSCATE continental-surface data services'
        ]
    ),
    financials=FinancialMetrics(
        tcv=1800000.0,
        annual_revenue=300000.0,
        project_margin=21.0,
        payment_terms='30 days from invoice date, quarterly invoicing'
    ),
    risks=[
        RiskAnalysis(
            risk_description='Penalties due to SLA non-compliance caused by quality or delivery delays',
            impact_level='high',
            mitigation_strategy='Rigorous QA and project management',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Cost overruns from underestimation of rework, missing packages, or non-representative
platforms',
            impact_level='medium',
            mitigation_strategy='Detailed due-diligence and contingency planning',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Competency retention issues due to turnover and declining activity',
            impact_level='medium',
            mitigation_strategy='Knowledge management and retention incentives'

In [ ]:
chunks = rag.chunck(rainbow_report)
debug(chunks)


In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

dyn_tool = rag.create_vector_search_tool()
debug(convert_to_openai_tool(dyn_tool))


In [ ]:
r = dyn_tool.invoke({"query": "CNES", "selected_sections": ["team"],"entity_keys": []})
print(r)

In [ ]:
# 2. Index the document
rag.store_chunks(chunks)
print("Document stored.")


In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore("revenue", k=2, filter={"field_name": {"$eq": "financials"}})
print("Vector hits:", hits)

In [ ]:
rag.get_key_description()

In [ ]:
#vector_store_factory.delete_collection()